In [1]:
!cd ../ && poetry install

Installing dependencies from lock file


The lock file might not be compatible with the current version of Poetry.
Upgrade Poetry to ensure the lock file is read properly or, alternatively, regenerate the lock file with the `poetry lock` command.



No dependencies to install or update

Installing the current project: rag-on-papers (0.1.0)
If you do not want to install the current project use --no-root.
If you want to use Poetry only for dependency management but not for packaging, you can disable package mode by setting package-mode = false in your pyproject.toml file.
In a future version of Poetry this warning will become an error!


# RAG on Papers — End-to-End Walkthrough on **Real Data**

This notebook runs the *actual* pipeline functions across **every module (1→8)**
using **real medical papers** pulled live from PubMed Central. Intermediate
artifacts are cached under `notebooks/data/` so re-runs are cheap (no re-fetch,
no re-embed).

It calls the same functions the Celery tasks call — just **synchronously and
inline** so you can see each stage's output.

### Prerequisites
- `cd docker && docker compose up -d` (Qdrant, MinIO, Grobid, Postgres, RabbitMQ,
  Redis, MLflow…). The notebook only *needs* **Qdrant** (retrieval/index) and
  **OpenRouter** (dense embeddings + agent); MinIO/Grobid are used by optional cells.
- A valid `OPENROUTER_API_KEY` and `NCBI_API_KEY` in `.env` (both already set).
- Run from the project root in the project env: `poetry run jupyter lab`.

### What each module contributes
| Module | Real work this notebook does |
|---|---|
| 1 fetch | live PMC `esearch` + `efetch` → raw JATS XML |
| 1b parse | `parse_pmc_xml` (XML path) + optional Grobid PDF path |
| 2 chunk | `chunk_paper` → real `Chunk`s + S3/MinIO round-trip |
| 3 embed | real SPLADE (local) + OpenRouter dense (512d) |
| 4 index | `create_collection` + `ingest_chunks` into live Qdrant |
| 5 retrieve | semantic / lexical / hybrid / tag on a real query |
| 6 rerank | `rerank` (RRF → MMR) over real candidates |
| 7 agent | full CrewAI crew (gated by a flag — costs tokens) |
| 8 eval | recall@k / MRR on the real corpus; optional RAGAS + MLflow |

## 0. Setup — paths, event loop, host networking

- Adds the repo root to `sys.path`.
- Applies `nest_asyncio` so the modules' internal `asyncio.run()` works inside Jupyter.
- **Remaps Docker hostnames → `localhost`** because we run on the host while the
  services run in containers (compose publishes their ports). If you instead run
  this notebook *inside* the compose network, comment out the remap block.

In [2]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "modules").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

# Allow nested event loops (modules call asyncio.run() internally)
try:
    import nest_asyncio

    nest_asyncio.apply()
    print("nest_asyncio applied")
except ImportError:
    print("WARNING: `pip install nest_asyncio` if you hit 'event loop already running'")

# Local data cache
DATA_DIR = ROOT / "notebooks" / "data"
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "parsed").mkdir(parents=True, exist_ok=True)
print("Data cache:", DATA_DIR)

Repo root: /Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers
nest_asyncio applied
Data cache: /Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/notebooks/data


In [3]:
# Remap container DNS names to localhost for host execution, and rebind the S3
# client to the local MinIO endpoint (so save/load_chunks_to_s3 work on the host).
import boto3
from botocore.client import Config

from shared import s3 as s3mod
from shared.config import settings

settings.qdrant_url = "http://localhost:6333"
settings.grobid_url = "http://localhost:8070"
settings.mlflow_tracking_uri = "http://localhost:5000"
settings.s3_endpoint_url = "http://localhost:9000"

s3mod._client = boto3.client(
    "s3",
    aws_access_key_id=settings.aws_access_key_id or "minioadmin",
    aws_secret_access_key=settings.aws_secret_access_key or "minioadmin",
    region_name=settings.aws_region,
    endpoint_url="http://localhost:9000",
    config=Config(s3={"addressing_style": "path"}),
)
print("qdrant ->", settings.qdrant_url, "| minio ->", settings.s3_endpoint_url)

qdrant -> http://localhost:6333 | minio -> http://localhost:9000


## 0b. Run configuration

Tune the corpus size and which expensive stages run. The CrewAI agent and RAGAS
make many LLM calls — leave them off until the rest works.

In [4]:
SEARCH_QUERY = "metformin type 2 diabetes renal function"  # PMC search
MAX_PAPERS = 6  # how many full-text papers to ingest
FORCE_REFRESH = False  # True = ignore caches and re-fetch/re-embed

RUN_AGENT = True  # Module 7: one full CrewAI run (costs tokens)
RUN_RAGAS = False  # Module 8: RAGAS metrics (many LLM calls)
RUN_ABLATION = False  # Module 8: full strategy ablation (very expensive)
SAMPLE_PDF_PATH = None  # set to a local .pdf to exercise the Grobid path

EMBEDDED_CACHE = DATA_DIR / "embedded_chunks.jsonl"

## Module 1 — Fetch (real PubMed Central API)

Uses the real `PubMedAdapter`: `search()` (esearch) returns PMC IDs, `fetch()`
(efetch) returns JATS XML. We fetch one ID at a time (efetch concatenates
multiple articles) and cache each XML under `data/raw/`. This is the same adapter
`fetch_single_paper` uses — we just skip the S3/Postgres/Celery glue.

In [5]:
import asyncio

from modules.module1_fetch.adapters import PubMedAdapter

adapter = PubMedAdapter()
pmc_ids = asyncio.run(adapter.search(SEARCH_QUERY, max_results=MAX_PAPERS * 3))
print(f"esearch returned {len(pmc_ids)} PMC ids; fetching up to {MAX_PAPERS} full texts")

raw_xmls = {}  # pmcid -> xml bytes
for pid in pmc_ids:
    if len(raw_xmls) >= MAX_PAPERS:
        break
    cache = DATA_DIR / "raw" / f"{pid}.xml"
    if cache.exists() and not FORCE_REFRESH:
        raw_xmls[pid] = cache.read_bytes()
        continue
    try:
        xml = asyncio.run(adapter.fetch(pid))
        if b"<article" in xml:  # keep only real articles
            cache.write_bytes(xml)
            raw_xmls[pid] = xml
            print(f"  fetched {pid} ({len(xml):,} bytes)")
    except Exception as e:
        print(f"  skip {pid}: {e}")

print(f"\nCached {len(raw_xmls)} raw XML files in {DATA_DIR / 'raw'}")

esearch returned 18 PMC ids; fetching up to 6 full texts


  fetched 13286198 (138,500 bytes)


  fetched 13284589 (116,677 bytes)


  fetched 13284594 (135,038 bytes)


  fetched 13284793 (7,911 bytes)


  fetched 13286652 (396,726 bytes)


  fetched 13284910 (11,162 bytes)

Cached 6 raw XML files in /Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/notebooks/data/raw


## Module 1b — Parse (real JATS XML → structured schema)

`parse_pmc_xml` (lxml) turns each raw XML into the uniform `structured.json`
schema every downstream module consumes. Parsed docs are cached under
`data/parsed/`. We keep only papers that actually have text.

In [6]:
import json

from modules.module1b_parse.pmc_parser import parse_pmc_xml
from shared.utils import doi_to_slug

papers = []  # list of structured dicts
for pid, xml in raw_xmls.items():
    try:
        structured = parse_pmc_xml(xml)
    except Exception as e:
        print(f"  parse failed {pid}: {e}")
        continue
    if not (structured.get("abstract") or structured.get("sections")):
        continue  # no usable text
    structured["source_db"] = "pubmed"
    structured["pmcid"] = pid
    slug = doi_to_slug(structured.get("doi") or pid)
    (DATA_DIR / "parsed" / f"{slug}.json").write_text(json.dumps(structured, indent=1))
    papers.append(structured)

print(f"Parsed {len(papers)} usable papers\n")
p = papers[0]
print("Example paper:")
print("  title  :", p["title"][:90])
print("  doi    :", p["doi"] or "(none)")
print("  journal:", p["journal"], "|", p["pub_date"])
print(
    "  sections:",
    len(p["sections"]),
    "| figures:",
    len(p["figures"]),
    "| tables:",
    len(p["tables"]),
)
print("  abstract:", p["abstract"][:200], "...")

Parsed 6 usable papers

Example paper:
  title  : Pharmacokinetic interactions between three SGLT2 inhibitors and telmisartan: A focus on em
  doi    : 10.1371/journal.pone.0350400
  journal: PLOS One | 2026-06-22
  sections: 30 | figures: 5 | tables: 4
  abstract: BackgroundPatients with type 2 diabetes mellitus (T2DM) usually accompany with the occurrence of high blood pressure, necessitating a combination of treatments. This includes sodium-glucose co-transpo ...


### Module 1b (optional) — Grobid PDF path + figure/table handlers

The XML path above never touches Grobid. To exercise the PDF branch
(`parse_pdf_to_tei` → `parse_tei` → `extract_and_describe_figure`), set
`SAMPLE_PDF_PATH` to a local PDF in the config cell. Guarded so it's skipped by
default and never breaks the run.

In [7]:
from modules.module1b_parse.grobid_client import grobid_is_alive, parse_pdf_to_tei
from modules.module1b_parse.tei_parser import parse_tei

if SAMPLE_PDF_PATH and grobid_is_alive():
    try:
        tei = parse_pdf_to_tei(SAMPLE_PDF_PATH)
        tei_doc = parse_tei(tei)
        print("Grobid parsed:", tei_doc["title"][:80])
        print("sections:", len(tei_doc["sections"]), "| figures:", len(tei_doc["figures"]))

        # Figure handler: crop + OpenRouter description + MinIO upload (first figure only)
        figs = [f for f in tei_doc["figures"] if f["coords"]["w"] > 0]
        if figs:
            from modules.module1b_parse.figure_handler import extract_and_describe_figure
            from modules.module1b_parse.table_handler import save_table_to_s3

            described = extract_and_describe_figure(SAMPLE_PDF_PATH, figs[0], "demo-grobid")
            print("\nfigure description:", described["description"][:200])
            for i, t in enumerate(tei_doc["tables"]):
                save_table_to_s3(t, "demo-grobid", i)
            print("tables saved to MinIO:", len(tei_doc["tables"]))
    except Exception as e:
        print("Grobid path error:", e)
else:
    print("Skipped Grobid PDF path (set SAMPLE_PDF_PATH and ensure Grobid is up).")

Skipped Grobid PDF path (set SAMPLE_PDF_PATH and ensure Grobid is up).


## Module 2 — Chunk (real `Chunk` objects)

`chunk_paper` produces abstract/section/figure/table chunks, each with an
injected contextual header. We verify the 512-token ceiling and round-trip one
paper's chunks through **MinIO** via the real `save_/load_chunks_to_s3`.

In [8]:
from modules.module2_chunk.chunker import (
    chunk_paper,
    count_tokens,
    load_chunks_from_s3,
    save_chunks_to_s3,
)

all_chunks = []
for p in papers:
    slug = doi_to_slug(p.get("doi") or p["pmcid"])
    all_chunks.extend(chunk_paper(p, s3_parsed_key=f"parsed/{slug}/structured.json"))

print(f"{len(all_chunks)} chunks from {len(papers)} papers")
by_type = {}
for c in all_chunks:
    by_type[c.element_type] = by_type.get(c.element_type, 0) + 1
print("by element_type:", by_type)
print(
    "max tokens in any chunk text:",
    max(count_tokens(c.text) for c in all_chunks),
    "(<= 512 for sections)",
)

print("\n--- sample header-injected chunk ---")
print(all_chunks[0].text_with_header[:400])

# Real MinIO round-trip for the first paper's chunks
doi0 = papers[0].get("doi") or papers[0]["pmcid"]
p0_chunks = [c for c in all_chunks if c.doi == papers[0].get("doi", "")] or all_chunks[:5]
try:
    key = save_chunks_to_s3(p0_chunks, doi0)
    reloaded = load_chunks_from_s3(doi0)
    print(f"\nMinIO round-trip OK: wrote {key}, reloaded {len(reloaded)} chunks")
except Exception as e:
    print("\nMinIO round-trip skipped:", e)

152 chunks from 6 papers
by element_type: {'abstract': 6, 'section': 119, 'figure': 15, 'table': 12}
max tokens in any chunk text: 7188 (<= 512 for sections)

--- sample header-injected chunk ---
Paper: Pharmacokinetic interactions between three SGLT2 inhibitors and telmisartan: A focus on empagliflozin, ertugliflozin, and henagliflozin
Journal: PLOS One  |  Date: 2026-06-22
Section: Abstract
---
BackgroundPatients with type 2 diabetes mellitus (T2DM) usually accompany with the occurrence of high blood pressure, necessitating a combination of treatments. This includes sodium-glucose co-tra



MinIO round-trip skipped: Connection was closed before we received a valid response from endpoint URL: "http://localhost:9000/rag-medical-papers/chunks/10.1371_journal.pone.0350400.jsonl".


## Module 3 — Embed (real dense + sparse vectors)

- **Sparse**: local FastEmbed SPLADE (downloads ~500 MB on first run, CPU-only).
- **Dense**: OpenRouter `text-embedding-3-large` truncated to **512 dims**.

Embedded chunks (vectors included) are cached to `data/embedded_chunks.jsonl`, so
later runs skip the API entirely.

In [9]:
import asyncio
import json

from modules.module3_embed.dense import embed_texts
from modules.module3_embed.sparse import generate_sparse_vectors
from shared.models import Chunk

if EMBEDDED_CACHE.exists() and not FORCE_REFRESH:
    all_chunks = [
        Chunk.model_validate_json(line) for line in EMBEDDED_CACHE.read_text().splitlines() if line
    ]
    print(f"Loaded {len(all_chunks)} embedded chunks from cache (no API calls)")
else:
    texts = [c.text_with_header for c in all_chunks]
    print("Generating sparse vectors (SPLADE, local)...")
    sparse = generate_sparse_vectors(texts)
    print("Generating dense vectors (OpenRouter, 512d)...")
    dense = asyncio.run(embed_texts(texts))
    for c, d, s in zip(all_chunks, dense, sparse, strict=False):
        c.dense_vector, c.sparse_indices, c.sparse_values = d, s["indices"], s["values"]
    EMBEDDED_CACHE.write_text("\n".join(c.model_dump_json() for c in all_chunks))
    print(f"Cached {len(all_chunks)} embedded chunks -> {EMBEDDED_CACHE}")

c0 = all_chunks[0]
print("\ndense dims:", len(c0.dense_vector), "| sparse non-zeros:", len(c0.sparse_indices))

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/.venv_rag_on_papers/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 240 embedded chunks from cache (no API calls)

dense dims: 512 | sparse non-zeros: 109


## Module 4 — Index (real Qdrant upsert)

`create_collection` (idempotent: dense HNSW + sparse + 9 payload indexes) then
`ingest_chunks` upserts every chunk. Re-running is safe — same `chunk_id`
overwrites.

In [10]:
from modules.module4_index.ingest import ingest_chunks
from modules.module4_index.setup import COLLECTION, create_collection, get_client

create_collection()
ingest_chunks(all_chunks)

info = get_client().get_collection(COLLECTION)
print(f"Qdrant '{COLLECTION}' now has {info.points_count} points")
corpus_dois = sorted({c.doi for c in all_chunks if c.doi})
print("corpus DOIs:", corpus_dois)

{"collection": "medical_papers", "event": "qdrant_collection_created", "level": "info", "timestamp": "2026-06-23T16:44:11.416968Z"}


{"field": "doi", "event": "qdrant_payload_index_created", "level": "info", "timestamp": "2026-06-23T16:44:11.434357Z"}


{"field": "pub_year", "event": "qdrant_payload_index_created", "level": "info", "timestamp": "2026-06-23T16:44:11.455983Z"}


{"field": "pub_date", "event": "qdrant_payload_index_created", "level": "info", "timestamp": "2026-06-23T16:44:11.469563Z"}


{"field": "element_type", "event": "qdrant_payload_index_created", "level": "info", "timestamp": "2026-06-23T16:44:11.483675Z"}


...

{"field": "has_figure", "event": "qdrant_payload_index_created", "level": "info", "timestamp": "2026-06-23T16:44:11.546417Z"}


/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/.venv_rag_on_papers/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.5. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


{"field": "has_table", "event": "qdrant_payload_index_created", "level": "info", "timestamp": "2026-06-23T16:44:11.562135Z"}


{"count": 240, "doi": "10.7759/cureus.109440", "event": "chunks_ingested", "level": "info", "timestamp": "2026-06-23T16:44:11.723568Z"}


Qdrant 'medical_papers' now has 240 points
corpus DOIs: ['10.1016/j.ajcnut.2025.06.015', '10.2147/DDDT.S612176', '10.7150/ijms.126520', '10.7150/ijms.133077', '10.7759/cureus.109350', '10.7759/cureus.109440']


## Module 5 — Retrieve (all four strategies, real Qdrant)

Same query through `SemanticRetriever`, `LexicalRetriever`, `HybridRetriever`,
and a **tag** run (hybrid + metadata filter).

In [11]:
from modules.module5_retrieve import HybridRetriever, LexicalRetriever, SemanticRetriever

QUERY = "What are the renal effects of long-term metformin use in type 2 diabetes?"


def show(name, results, n=3):
    print(f"\n[{name}] {len(results)} hits")
    for r in results[:n]:
        print(
            f"  {r.score:.4f}  {r.payload.get('section_heading', '')[:40]:40}  {r.payload.get('text', '')[:70]}"
        )


show("semantic", SemanticRetriever().retrieve(QUERY, top_k=20))
show("lexical", LexicalRetriever().retrieve(QUERY, top_k=20))
show("hybrid", HybridRetriever().retrieve(QUERY, top_k=20))

# Tag retrieval = hybrid + filter (only 'section' chunks here)
tagged = HybridRetriever().retrieve(QUERY, top_k=20, filters={"element_type": "section"})
show("tag (element_type=section)", tagged)


[semantic] 20 hits
  0.6482  Discussion                                Metformin has been used for over 70 years in the treatment of diabetes
  0.5973  Effects of genetic variation in glucose-  At the drug overall level, pooled analyses revealed that Metformin sig
  0.5904  Effects of genetic variation in glucose-  Based on HbA1c-derived IVs,  GPD1  (Metformin) significantly reduced C


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 44243.71it/s]


[lexical] 20 hits
  17.1227  Discussion                                Metformin has been used for over 70 years in the treatment of diabetes
  16.9335  3.8 Therapeutic interventions and sepsis  It is important to note that although metformin is not primarily indic
  16.0906  3.8 Therapeutic interventions and sepsis  Metformin is widely recommended as a first-line therapy for the manage



[hybrid] 20 hits
  1.0000  Discussion                                Metformin has been used for over 70 years in the treatment of diabetes
  0.4583  Effects of genetic variation in glucose-  At the drug overall level, pooled analyses revealed that Metformin sig
  0.4167  3.8 Therapeutic interventions and sepsis  It is important to note that although metformin is not primarily indic



[tag (element_type=section)] 20 hits
  1.0000  Discussion                                Metformin has been used for over 70 years in the treatment of diabetes
  0.4583  Effects of genetic variation in glucose-  At the drug overall level, pooled analyses revealed that Metformin sig
  0.4167  3.8 Therapeutic interventions and sepsis  Metformin is widely recommended as a first-line therapy for the manage


## Module 6 — Rerank (real RRF → MMR)

`rerank` fuses the retriever outputs with RRF, fetches their dense vectors back
from Qdrant, then applies MMR for a diverse final 8.

In [12]:
import asyncio

from modules.module3_embed.dense import embed_texts
from modules.module6_rerank.pipeline import rerank

retrieval_map = {
    "semantic": SemanticRetriever().retrieve(QUERY, top_k=50),
    "lexical": LexicalRetriever().retrieve(QUERY, top_k=50),
}
query_vec = asyncio.run(embed_texts([QUERY]))[0]
final = rerank(QUERY, query_vec, retrieval_map, rrf_top_n=20, mmr_top_k=8)

print(f"Final {len(final)} reranked chunks (RRF -> MMR):")
for r in final:
    print(
        f"  {r.payload.get('doi', '')[:24]:24}  {r.payload.get('section_heading', '')[:30]:30}  {r.payload.get('text', '')[:60]}"
    )

Final 8 reranked chunks (RRF -> MMR):
  10.7150/ijms.133077       Discussion                      Metformin has been used for over 70 years in the treatment o
  10.7150/ijms.126520       3.8 Therapeutic interventions   Metformin is widely recommended as a first-line therapy for 
  10.1016/j.ajcnut.2025.06  Recent kidney outcomes with GL  The Evaluate Renal Function with Semaglutide Once Weekly (FL
  10.2147/DDDT.S612176      Metformin and Semaglutide (Rep  Metformin, widely used for type 2 diabetes, has garnered int
  10.7150/ijms.133077       Effects of genetic variation i  At the drug overall level, pooled analyses revealed that Met
  10.7150/ijms.133077       Effects of genetic variation i  Based on HbA1c-derived IVs,  GPD1  (Metformin) significantly
  10.7150/ijms.133077       Discussion                      . Consistent with this, our SMR analysis confirmed that gene
  10.7150/ijms.133077       Discussion                      We also identified some adverse effects associated with

## Module 7 — RAG Agent (real CrewAI crew)

`run_query` runs the 3-agent crew (retrieval → analysis → synthesis) against the
live corpus and OpenRouter, returning a cited answer. **Gated by `RUN_AGENT`** —
this makes many LLM calls.

In [13]:
if RUN_AGENT:
    from modules.module7_agent.crew import run_query

    try:
        answer = run_query(QUERY)
        print("\n=== FINAL ANSWER ===\n")
        print(answer)
    except Exception as e:
        print("Agent run failed:", e)
else:
    print("RUN_AGENT is False — set it True to run the full crew.")

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/.venv_rag_on_papers/lib/python3.12/site-packages/crewai/routers/router.py:12: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class Router(BaseModel):


# Agent: Medical Research Retrieval Specialist
## Task: User question: "What are the renal effects of long-term metformin use in type 2 diabetes?"
Rewrite into an optimal retrieval query. Select the best strategy (semantic|lexical|hybrid|tag) and any relevant MeSH term or date filters. Use multi_hop_retrieve_tool if the question requires chained reasoning across multiple topics.


/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/modules/module7_agent/tools.py:84: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use invoke instead.
  chunk_text = retrieve_tool(current_query, strategy="hybrid")


 

I encountered an error while trying to use the tool. This was the error: BaseTool.__call__() got an unexpected keyword argument 'strategy'.
 Tool multi_hop_retrieve_tool accepts these inputs: Multi-hop retrieval for questions requiring chained reasoning.
Each hop generates a sub-query from the previous hop's context.



# Agent: Medical Research Retrieval Specialist
## Thought: Thought: To address the user's question on the renal effects of long-term metformin use in type 2 diabetes, I should consider MeSH terms related to "Metformin", "Type 2 Diabetes Mellitus", and "Kidney". Considering the complexity and specificity of the query, "long-term metformin use", it may be beneficial to use the multi-hop retrieval tool to gather comprehensive information, potentially covering effects over different time frames or involving different mechanisms.
## Using tool: multi_hop_retrieve_tool
## Tool Input: 
"{\"initial_query\": \"renal effects of long-term metformin use in type 2 diabetes\"}"
##

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/.venv_rag_on_papers/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.5. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


...

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/.venv_rag_on_papers/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.5. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(




# Agent: Medical Evidence Analyst
## Thought: The tool did not provide the desired data. I need to try another retrieval attempt, focusing on slightly adjusted criteria or parameters to ensure relevant literature is obtained.
## Using tool: retrieve_tool
## Tool Input: 
"{\"query\": \"renal effects of metformin in type 2 diabetes patients\", \"strategy\": \"hybrid\", \"mesh_terms\": [\"metformin\", \"type 2 diabetes\", \"renal safety\", \"kidney function\"], \"year_from\": 2010, \"top_k\": 100}"
## Tool Output: 



You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:

Tool Name: retrieve_tool(query: str, strategy: str = 'hybrid', mesh_terms: list[str] | None = None, year_from: int | None = None, element_type: str | None = None, top_k: int = 100) -> str
Tool Description: Retrieve relevant chunks from the medical research corpus.
strategy: semantic | lexical | hybrid | tag
Returns top chunks with DOI, section, and text.
Tool Arguments: {



# Agent: Medical Evidence Analyst
## Final Answer: 
my best complete final answer to the task.
# Agent: Medical Research Synthesiser
## Task: Answer the original question: "What are the renal effects of long-term metformin use in type 2 diabetes?"
Use only the retrieved evidence. Cite every claim as [DOI, Section]. If evidence is insufficient, state this explicitly.




# Agent: Medical Research Synthesiser
## Final Answer: 
The renal effects of long-term metformin use in type 2 diabetes are nuanced, and recent studies have provided important insights into its safety and implications for kidney function. Metformin is primarily excreted unchanged by the kidneys, which has historically raised concerns about its use in patients with renal impairment, due to the risk of lactic acidosis [DOI unavailable, Context provided].

Recent guidelines and studies, however, suggest that metformin may be safe even in patients with mild to moderate renal impairment. Several studies indicate that metformin use in these patients is associated with reduced mortality without significant renal adverse effects [DOI unavailable, Context provided].

The key factors influencing these outcomes include metformin's primary action of reducing hepatic glucose production and its secondary effects on gut microbiota, which may indirectly influence renal health over time [DOI unavaila


=== FINAL ANSWER ===

The renal effects of long-term metformin use in type 2 diabetes are nuanced, and recent studies have provided important insights into its safety and implications for kidney function. Metformin is primarily excreted unchanged by the kidneys, which has historically raised concerns about its use in patients with renal impairment, due to the risk of lactic acidosis [DOI unavailable, Context provided].

Recent guidelines and studies, however, suggest that metformin may be safe even in patients with mild to moderate renal impairment. Several studies indicate that metformin use in these patients is associated with reduced mortality without significant renal adverse effects [DOI unavailable, Context provided].

The key factors influencing these outcomes include metformin's primary action of reducing hepatic glucose production and its secondary effects on gut microbiota, which may indirectly influence renal health over time [DOI unavailable, Context provided]. Although tr

## Module 8 — Evaluation (real retrieval metrics)

`recall_at_k` / `mean_reciprocal_rank` over the real corpus: we treat the ingested
papers' DOIs as the relevant set for the topical query. RAGAS and the MLflow
ablation are gated (expensive, LLM-judged).

In [14]:
from modules.module8_eval.retrieval_metrics import (
    mean_reciprocal_rank,
    recall_at_k,
    run_retrieval_and_rerank,
)

retrieved = run_retrieval_and_rerank(QUERY, strategy="hybrid")
print("retrieved", len(retrieved), "reranked chunks")
print("recall@8  :", round(recall_at_k(retrieved, corpus_dois, k=8), 3))
print("MRR       :", round(mean_reciprocal_rank(retrieved, corpus_dois), 3))

if RUN_RAGAS:
    from modules.module8_eval.ragas_eval import evaluate_strategy

    test_set = [{"query": QUERY, "expected_answer": "", "source_dois": corpus_dois}]
    print("\nRAGAS:", evaluate_strategy(test_set, "hybrid"))

if RUN_ABLATION:
    from modules.module8_eval.ablation import run_ablation

    run_ablation([{"query": QUERY, "expected_answer": "", "source_dois": corpus_dois}])
    print("Ablation logged to MLflow at", settings.mlflow_tracking_uri)

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Projects/rag-on-papers/.venv_rag_on_papers/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.5. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


retrieved 8 reranked chunks
recall@8  : 0.667
MRR       : 1.0


## Where the data lives & cleanup

- `notebooks/data/raw/*.xml` — raw JATS XML per PMC id
- `notebooks/data/parsed/*.json` — structured papers
- `notebooks/data/embedded_chunks.jsonl` — chunks **with** dense+sparse vectors (re-run cache)
- Qdrant collection `medical_papers` — the live index (persists in the `qdrant_data` volume)
- MinIO bucket `rag-medical-papers` — chunk JSONL + any Grobid figures/tables

**Re-run cheaply:** everything is cached; set `FORCE_REFRESH = True` to rebuild.

**Reset the index:**
```python
from modules.module4_index.setup import delete_collection
delete_collection()
```

`notebooks/data/` is gitignored so real fetched content is never committed.